In [0]:
from pyspark.sql import functions as F

def load(name):
    return spark.table(f"workspace.default.{name}")

outcomes = (load("outcomes")
            .withColumn("week", F.col("week").cast("int"))
            .withColumn("disrupted", F.col("disrupted").cast("int")))

# 1) disruption flag in the last history week
last = outcomes.filter(F.col("week") == 208).select("material", "disrupted")

# 2) most recent week each material was NOT disrupted
last_ok = (outcomes.filter(F.col("disrupted") == 0)
           .groupBy("material")
           .agg(F.max("week").alias("last_ok_week")))

# 3) weeks the current disruption has lasted so far (0 if not disrupted)
state = (last.join(last_ok, "material", "left")
         .withColumn("weeks_in_run",
                     F.when(F.col("disrupted") == 1, 208 - F.col("last_ok_week"))
                      .otherwise(0)))

display(state.orderBy("material"))


In [0]:
from pyspark.sql import functions as F

LAST_WEEK = 208          # change to 221 when the new data is uploaded

outcomes = (spark.table("workspace.default.outcomes")
            .withColumn("week", F.col("week").cast("int"))
            .withColumn("disrupted", F.col("disrupted").cast("int")))

# is each material disrupted in the last week?
last = (outcomes.filter(F.col("week") == LAST_WEEK)
        .select("material", "disrupted"))

# the most recent healthy week for each material
last_ok = (outcomes.filter(F.col("disrupted") == 0)
           .groupBy("material")
           .agg(F.max("week").alias("last_ok_week")))

# how many weeks the current disruption has lasted
state = (last.join(last_ok, "material", "left")
         .withColumn("weeks_in_run",
                     F.when(F.col("disrupted") == 1, LAST_WEEK - F.col("last_ok_week"))
                      .otherwise(0)))

display(state.orderBy("material"))

In [0]:
display(spark.table("workspace.default.sample_submission"))